# Online Retail II — Data Cleaning

This notebook transforms the raw Online Retail II dataset into documented, analysis-ready transaction datasets while preserving cancellations and operational adjustments separately.

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.2f}".format)

In [2]:
project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

raw_data_path = (
    project_root
    / "data"
    / "raw"
    / "online_retail_II.xlsx"
)

print("Raw dataset exists:", raw_data_path.exists())
print("Raw dataset path:", raw_data_path)

Raw dataset exists: True
Raw dataset path: c:\Users\hp\Documents\Customer-Revenue-Retention-Analytics\data\raw\online_retail_II.xlsx


In [3]:
def load_raw_retail_data(file_path):
    workbook = pd.read_excel(
        file_path,
        sheet_name=None
    )

    combined_data = pd.concat(
        [
            dataframe.assign(SourceYear=sheet_name)
            for sheet_name, dataframe in workbook.items()
        ],
        ignore_index=True
    )

    return combined_data

In [4]:
retail_raw = load_raw_retail_data(raw_data_path)

print("Raw dataset shape:", retail_raw.shape)
display(retail_raw.head())

Raw dataset shape: (1067371, 9)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country,SourceYear
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,"13,085.00",United Kingdom,Year 2009-2010
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom,Year 2009-2010
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom,Year 2009-2010
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,"13,085.00",United Kingdom,Year 2009-2010
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,"13,085.00",United Kingdom,Year 2009-2010


## Schema Standardization and Duplicate Removal

In [5]:
retail_clean = retail_raw.copy()

business_columns = [
    "Invoice",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "Price",
    "Customer ID",
    "Country",
]

rows_before = len(retail_clean)

# Duplicates where every column, including SourceYear, matches
same_source_duplicates = retail_clean.duplicated(
    keep="first"
).sum()

# Duplicates based on the actual transaction fields
all_business_duplicates = retail_clean.duplicated(
    subset=business_columns,
    keep="first",
).sum()

# Additional duplicates caused by the worksheet overlap
cross_source_duplicates_removed = (
    all_business_duplicates
    - same_source_duplicates
)

# Remove duplicates while ignoring SourceYear
retail_clean = (
    retail_clean
    .drop_duplicates(
        subset=business_columns,
        keep="last",
    )
    .reset_index(drop=True)
)

rows_after = len(retail_clean)

remaining_business_duplicates = (
    retail_clean.duplicated(
        subset=business_columns
    ).sum()
)

column_mapping = {
    "Invoice": "invoice_no",
    "StockCode": "stock_code",
    "Description": "description",
    "Quantity": "quantity",
    "InvoiceDate": "invoice_date",
    "Price": "unit_price",
    "Customer ID": "customer_id",
    "Country": "country",
    "SourceYear": "source_year",
}

retail_clean = retail_clean.rename(
    columns=column_mapping
)

print("Rows before duplicate removal:", rows_before)
print(
    "Same-source duplicates removed:",
    same_source_duplicates,
)
print(
    "Cross-source duplicates removed:",
    cross_source_duplicates_removed,
)
print(
    "Total duplicate rows removed:",
    rows_before - rows_after,
)
print("Rows after duplicate removal:", rows_after)
print(
    "Remaining business duplicates:",
    remaining_business_duplicates,
)
print("\nStandardized columns:")
print(retail_clean.columns.tolist())

Rows before duplicate removal: 1067371
Same-source duplicates removed: 12133
Cross-source duplicates removed: 22202
Total duplicate rows removed: 34335
Rows after duplicate removal: 1033036
Remaining business duplicates: 0

Standardized columns:
['invoice_no', 'stock_code', 'description', 'quantity', 'invoice_date', 'unit_price', 'customer_id', 'country', 'source_year']


In [6]:
retail_clean["invoice_no"] = (
    retail_clean["invoice_no"]
    .astype("string")
    .str.strip()
    .str.upper()
)

retail_clean["stock_code"] = (
    retail_clean["stock_code"]
    .astype("string")
    .str.strip()
    .str.upper()
)

retail_clean["description"] = (
    retail_clean["description"]
    .astype("string")
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
    .str.upper()
)

retail_clean["country"] = (
    retail_clean["country"]
    .astype("string")
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

retail_clean["source_year"] = (
    retail_clean["source_year"]
    .astype("string")
    .str.strip()
)

retail_clean["customer_id"] = (
    retail_clean["customer_id"]
    .astype("Int64")
)

retail_clean["invoice_date"] = pd.to_datetime(
    retail_clean["invoice_date"],
    errors="coerce"
)

retail_clean["quantity"] = pd.to_numeric(
    retail_clean["quantity"],
    errors="coerce"
)

retail_clean["unit_price"] = pd.to_numeric(
    retail_clean["unit_price"],
    errors="coerce"
)

retail_clean.info(show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 1033036 entries, 0 to 1033035
Data columns (total 9 columns):
 #   Column        Non-Null Count    Dtype         
---  ------        --------------    -----         
 0   invoice_no    1033036 non-null  string        
 1   stock_code    1033036 non-null  string        
 2   description   1028761 non-null  string        
 3   quantity      1033036 non-null  int64         
 4   invoice_date  1033036 non-null  datetime64[us]
 5   unit_price    1033036 non-null  float64       
 6   customer_id   797885 non-null   Int64         
 7   country       1033036 non-null  string        
 8   source_year   1033036 non-null  string        
dtypes: Int64(1), datetime64[us](1), float64(1), int64(1), string(5)
memory usage: 71.9 MB


## Transaction Classification

In [7]:
retail_clean["transaction_type"] = "Sale"

retail_clean.loc[
    retail_clean["unit_price"].eq(0),
    "transaction_type"
] = "Zero-Price Transaction"

retail_clean.loc[
    retail_clean["quantity"].lt(0),
    "transaction_type"
] = "Stock Adjustment"

retail_clean.loc[
    retail_clean["unit_price"].lt(0),
    "transaction_type"
] = "Accounting Adjustment"

retail_clean.loc[
    retail_clean["invoice_no"].str.startswith(
        "C",
        na=False
    ),
    "transaction_type"
] = "Cancellation"

retail_clean["line_value"] = (
    retail_clean["quantity"]
    * retail_clean["unit_price"]
)

retail_clean["transaction_type"] = (
    retail_clean["transaction_type"]
    .astype("category")
)

In [8]:
transaction_summary = (
    retail_clean
    .groupby(
        "transaction_type",
        observed=True
    )
    .agg(
        row_count=("invoice_no", "size"),
        unique_invoices=("invoice_no", "nunique"),
        total_quantity=("quantity", "sum"),
        net_line_value=("line_value", "sum")
    )
    .reset_index()
)

transaction_summary["percentage_of_rows"] = (
    transaction_summary["row_count"]
    / len(retail_clean)
    * 100
).round(2)

transaction_summary = transaction_summary.sort_values(
    by="row_count",
    ascending=False
)

print("Cleaned dataset shape:", retail_clean.shape)
display(transaction_summary)

Cleaned dataset shape: (1033036, 11)


,transaction_type,row_count,unique_invoices,total_quantity,net_line_value,percentage_of_rows
2,Sale,1007913,40077,11205148,"20,476,260.45",97.57
1,Cancellation,19104,8292,-476819,"-1,462,050.61",1.85
3,Stock Adjustment,3393,3393,-569314,0.00,0.33
4,Zero-Price Transaction,2621,1986,250759,0.00,0.25
0,Accounting Adjustment,5,5,5,"-158,676.14",0.00


## Analytical Data Layers

The standardized dataset is separated into sales, identified-customer sales, cancellations, and operational-adjustment layers. This allows each business question to use the appropriate records.

In [9]:
sales_transactions = retail_clean.loc[
    retail_clean["transaction_type"].eq("Sale")
].copy()

sales_transactions["sales_value"] = (
    sales_transactions["quantity"]
    * sales_transactions["unit_price"]
)

customer_sales = sales_transactions.dropna(
    subset=["customer_id"]
).copy()

cancellations = retail_clean.loc[
    retail_clean["transaction_type"].eq("Cancellation")
].copy()

cancellations["cancellation_value"] = (
    cancellations["quantity"].abs()
    * cancellations["unit_price"].abs()
)

operational_adjustments = retail_clean.loc[
    retail_clean["transaction_type"].isin([
        "Stock Adjustment",
        "Accounting Adjustment",
        "Zero-Price Transaction"
    ])
].copy()

In [10]:
layer_summary = pd.DataFrame({
    "Data Layer": [
        "All standardized transactions",
        "Positive sales transactions",
        "Identified-customer sales",
        "Cancellations",
        "Operational adjustments"
    ],
    "Row Count": [
        len(retail_clean),
        len(sales_transactions),
        len(customer_sales),
        len(cancellations),
        len(operational_adjustments)
    ],
    "Unique Invoices": [
        retail_clean["invoice_no"].nunique(),
        sales_transactions["invoice_no"].nunique(),
        customer_sales["invoice_no"].nunique(),
        cancellations["invoice_no"].nunique(),
        operational_adjustments["invoice_no"].nunique()
    ],
    "Missing Customer IDs": [
        retail_clean["customer_id"].isna().sum(),
        sales_transactions["customer_id"].isna().sum(),
        customer_sales["customer_id"].isna().sum(),
        cancellations["customer_id"].isna().sum(),
        operational_adjustments["customer_id"].isna().sum()
    ],
    "Missing Descriptions": [
        retail_clean["description"].isna().sum(),
        sales_transactions["description"].isna().sum(),
        customer_sales["description"].isna().sum(),
        cancellations["description"].isna().sum(),
        operational_adjustments["description"].isna().sum()
    ]
})

print(
    "Positive sales value:",
    f"£{sales_transactions['sales_value'].sum():,.2f}"
)

print(
    "Cancellation value:",
    f"£{cancellations['cancellation_value'].sum():,.2f}"
)

display(layer_summary)

Positive sales value: £20,476,260.45
Cancellation value: £1,462,797.75


,Data Layer,Row Count,Unique Invoices,Missing Customer IDs,Missing Descriptions
0,All standardized transactions,1033036,53628,235151,4275
1,Positive sales transactions,1007913,40077,228488,0
2,Identified-customer sales,779425,36969,0,0
3,Cancellations,19104,8292,714,0
4,Operational adjustments,6019,5384,5949,4275


## Customer Identification Coverage

In [11]:
coverage_data = sales_transactions.assign(
    customer_status=(
        sales_transactions["customer_id"]
        .notna()
        .map({
            True: "Identified Customer",
            False: "Anonymous Customer"
        })
    )
)

customer_coverage = (
    coverage_data
    .groupby(
        "customer_status",
        observed=True
    )
    .agg(
        row_count=("invoice_no", "size"),
        unique_invoices=("invoice_no", "nunique"),
        sales_value=("sales_value", "sum")
    )
    .reset_index()
)

customer_coverage["percentage_of_rows"] = (
    customer_coverage["row_count"]
    / len(sales_transactions)
    * 100
).round(2)

customer_coverage["percentage_of_sales_value"] = (
    customer_coverage["sales_value"]
    / sales_transactions["sales_value"].sum()
    * 100
).round(2)

display(customer_coverage)

,customer_status,row_count,unique_invoices,sales_value,percentage_of_rows,percentage_of_sales_value
0,Anonymous Customer,228488,3108,"3,101,456.18",22.67,15.15
1,Identified Customer,779425,36969,"17,374,804.27",77.33,84.85


## Identification of Non-Merchandise Sales Lines

In [12]:
standard_product_pattern = r"^\d{5}[A-Z]{0,2}$"

nonstandard_sales = sales_transactions.loc[
    ~sales_transactions["stock_code"].str.fullmatch(
        standard_product_pattern,
        na=False
    )
].copy()

nonstandard_summary = (
    nonstandard_sales
    .groupby(
        ["stock_code", "description"],
        dropna=False
    )
    .agg(
        row_count=("invoice_no", "size"),
        unique_invoices=("invoice_no", "nunique"),
        total_quantity=("quantity", "sum"),
        sales_value=("sales_value", "sum")
    )
    .reset_index()
    .sort_values(
        by="sales_value",
        ascending=False
    )
)

print("Non-standard positive sales rows:", len(nonstandard_sales))
print(
    "Non-standard sales value:",
    f"£{nonstandard_sales['sales_value'].sum():,.2f}"
)

display(nonstandard_summary.head(40))

Non-standard positive sales rows: 4699
Non-standard sales value: £833,568.30


,stock_code,description,row_count,unique_invoices,total_quantity,sales_value
32,M,MANUAL,851,788,9634,"339,241.29"
24,DOT,DOTCOM POSTAGE,1415,1415,1415,"309,854.11"
34,POST,POSTAGE,1851,1851,5363,"125,682.42"
3,AMAZONFEE,AMAZON FEE,3,3,3,"20,467.80"
6,C2,CARRIAGE,267,267,268,"13,426.00"
4,B,ADJUST BAD DEBT,1,1,1,"11,062.06"
1,ADJUST,ADJUSTMENT BY JOHN ON 26/01/2010 17,16,16,16,"6,563.50"
0,ADJUST,ADJUSTMENT BY JOHN ON 26/01/2010 16,20,20,20,"2,334.43"
2,ADJUST2,ADJUSTMENT BY PETER ON JUN 25 2010,3,3,3,731.05
27,GIFT_0001_30,DOTCOMGIFTSHOP GIFT VOUCHER £30.00,24,24,24,610.09


In [13]:
service_keyword_pattern = (
    r"POSTAGE|CARRIAGE|BANK CHARGE|AMAZON|"
    r"MANUAL|COMMISSION|FEE|SAMPLE|TEST|ADJUST"
)

service_candidates = (
    sales_transactions.loc[
        sales_transactions["description"].str.contains(
            service_keyword_pattern,
            case=False,
            na=False,
            regex=True
        )
    ]
    .groupby(
        ["stock_code", "description"],
        dropna=False
    )
    .agg(
        row_count=("invoice_no", "size"),
        unique_invoices=("invoice_no", "nunique"),
        sales_value=("sales_value", "sum")
    )
    .reset_index()
    .sort_values(
        by="sales_value",
        ascending=False
    )
)

display(service_candidates)

,stock_code,description,row_count,unique_invoices,sales_value
51,M,MANUAL,851,788,"339,241.29"
50,DOT,DOTCOM POSTAGE,1415,1415,"309,854.11"
52,POST,POSTAGE,1851,1851,"125,682.42"
22,23243,SET OF TEA COFFEE SUGAR TINS PANTRY,471,470,"23,489.42"
46,AMAZONFEE,AMAZON FEE,3,3,"20,467.80"
0,20748,KENSINGTON COFFEE SET,183,183,"14,266.98"
49,C2,CARRIAGE,267,267,"13,426.00"
2,21216,"SET 3 RETROSPOT TEA,COFFEE,SUGAR",503,498,"12,764.91"
15,22303,COFFEE MUG APPLES DESIGN,541,540,"12,701.63"
13,22301,COFFEE MUG CAT + BIRD DESIGN,645,643,"12,124.78"


## Controlled Merchandise Classification

Non-standard stock codes contain a combination of genuine products, shipping charges, fees, manual entries, discounts, samples, and gift vouchers. A controlled mapping is used instead of automatically excluding every non-standard code.

The earlier keyword search was exploratory only. For example, the substring `FEE` also occurs in `COFFEE`, so keyword matches are not used as the final exclusion rule.

In [14]:
shipping_codes = {
    "DOT",
    "POST",
    "C2"
}

fee_codes = {
    "AMAZONFEE",
    "BANK CHARGES",
    "CRUK"
}

manual_adjustment_codes = {
    "M",
    "B",
    "ADJUST",
    "ADJUST2",
    "TEST001",
    "TEST002"
}

retail_clean["line_category"] = "Merchandise"

retail_clean.loc[
    retail_clean["stock_code"].isin(shipping_codes),
    "line_category"
] = "Shipping or Service Charge"

retail_clean.loc[
    retail_clean["stock_code"].isin(fee_codes),
    "line_category"
] = "Platform or Bank Fee"

retail_clean.loc[
    retail_clean["stock_code"].isin(
        manual_adjustment_codes
    ),
    "line_category"
] = "Manual or Accounting Entry"

retail_clean.loc[
    retail_clean["stock_code"].eq("D"),
    "line_category"
] = "Discount Line"

retail_clean.loc[
    retail_clean["stock_code"].eq("S"),
    "line_category"
] = "Sample"

retail_clean.loc[
    retail_clean["stock_code"].str.startswith(
        "GIFT_",
        na=False
    ),
    "line_category"
] = "Gift Voucher"

retail_clean["line_category"] = (
    retail_clean["line_category"]
    .astype("category")
)

In [15]:
sales_transactions = retail_clean.loc[
    retail_clean["transaction_type"].eq("Sale")
].copy()

sales_transactions["sales_value"] = (
    sales_transactions["quantity"]
    * sales_transactions["unit_price"]
)

merchandise_sales = sales_transactions.loc[
    sales_transactions["line_category"].eq(
        "Merchandise"
    )
].copy()

non_merchandise_sales = sales_transactions.loc[
    ~sales_transactions["line_category"].eq(
        "Merchandise"
    )
].copy()

customer_merchandise_sales = (
    merchandise_sales
    .dropna(subset=["customer_id"])
    .copy()
)

In [16]:
line_category_summary = (
    sales_transactions
    .groupby(
        "line_category",
        observed=True
    )
    .agg(
        row_count=("invoice_no", "size"),
        unique_invoices=("invoice_no", "nunique"),
        total_quantity=("quantity", "sum"),
        sales_value=("sales_value", "sum")
    )
    .reset_index()
)

line_category_summary["percentage_of_sales_value"] = (
    line_category_summary["sales_value"]
    / sales_transactions["sales_value"].sum()
    * 100
).round(2)

line_category_summary = (
    line_category_summary
    .sort_values(
        by="sales_value",
        ascending=False
    )
)

assert (
    len(merchandise_sales)
    + len(non_merchandise_sales)
    == len(sales_transactions)
)

print(
    "Merchandise sales rows:",
    len(merchandise_sales)
)

print(
    "Identified-customer merchandise rows:",
    len(customer_merchandise_sales)
)

print(
    "Merchandise sales value:",
    f"£{merchandise_sales['sales_value'].sum():,.2f}"
)

display(line_category_summary)

Merchandise sales rows: 1003357
Identified-customer merchandise rows: 776596
Merchandise sales value: £19,643,861.64


,line_category,row_count,unique_invoices,total_quantity,sales_value,percentage_of_sales_value
3,Merchandise,1003357,39516,11188061,"19,643,861.64",95.93
6,Shipping or Service Charge,3533,3533,7046,"448,962.53",2.19
2,Manual or Accounting Entry,901,838,9725,"360,158.33",1.76
4,Platform or Bank Fee,37,36,37,"20,987.04",0.10
1,Gift Voucher,77,70,80,"1,756.17",0.01
0,Discount Line,5,5,196,397.89,0.00
5,Sample,3,3,3,136.85,0.00


## Merchandise Cancellation and Outlier Validation

Gross merchandise sales and cancellation values are measured separately. Extreme transactions are reviewed individually before any decision is made to retain or exclude them.

In [17]:
cancellations = retail_clean.loc[
    retail_clean["transaction_type"].eq(
        "Cancellation"
    )
].copy()

cancellations["cancellation_value"] = (
    cancellations["quantity"].abs()
    * cancellations["unit_price"].abs()
)

merchandise_cancellations = cancellations.loc[
    cancellations["line_category"].eq(
        "Merchandise"
    )
].copy()

gross_merchandise_value = (
    merchandise_sales["sales_value"].sum()
)

merchandise_cancellation_value = (
    merchandise_cancellations[
        "cancellation_value"
    ].sum()
)

estimated_net_merchandise_value = (
    gross_merchandise_value
    - merchandise_cancellation_value
)

print(
    "Gross merchandise value:",
    f"£{gross_merchandise_value:,.2f}"
)

print(
    "Merchandise cancellation value:",
    f"£{merchandise_cancellation_value:,.2f}"
)

print(
    "Estimated net merchandise value:",
    f"£{estimated_net_merchandise_value:,.2f}"
)

print(
    "Merchandise cancellation rows:",
    len(merchandise_cancellations)
)

Gross merchandise value: £19,643,861.64
Merchandise cancellation value: £716,462.57
Estimated net merchandise value: £18,927,399.07
Merchandise cancellation rows: 17915


In [18]:
outlier_columns = [
    "invoice_no",
    "stock_code",
    "description",
    "quantity",
    "invoice_date",
    "unit_price",
    "customer_id",
    "country",
    "sales_value"
]

print("Largest merchandise quantities:")
display(
    merchandise_sales
    .nlargest(10, "quantity")[
        outlier_columns
    ]
)

print("Highest merchandise unit prices:")
display(
    merchandise_sales
    .nlargest(10, "unit_price")[
        outlier_columns
    ]
)

print("Highest merchandise line values:")
display(
    merchandise_sales
    .nlargest(10, "sales_value")[
        outlier_columns
    ]
)

Largest merchandise quantities:


,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country,sales_value
1031554,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2011-12-09 09:15:00,2.08,16446,United Kingdom,"168,469.60"
557400,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,2011-01-18 10:01:00,1.04,12346,United Kingdom,"77,183.60"
89849,497946,37410,BLACK AND WHITE PAISLEY FLOWER MUG,19152,2010-02-15 11:57:00,0.10,13902,Denmark,"1,915.20"
125789,501534,21099,SET/6 STRAWBERRY PAPER CUPS,12960,2010-03-17 13:09:00,0.10,13902,Denmark,"1,296.00"
125791,501534,21091,SET/6 WOODLAND PAPER PLATES,12960,2010-03-17 13:09:00,0.10,13902,Denmark,"1,296.00"
125792,501534,21085,SET/6 WOODLAND PAPER CUPS,12744,2010-03-17 13:09:00,0.10,13902,Denmark,"1,274.40"
125790,501534,21092,SET/6 STRAWBERRY PAPER PLATES,12480,2010-03-17 13:09:00,0.10,13902,Denmark,"1,248.00"
133513,502269,21984,PACK OF 12 PINK PAISLEY TISSUES,10000,2010-03-23 15:36:00,0.25,17940,United Kingdom,"2,500.00"
133514,502269,21982,PACK OF 12 SUKI TISSUES,10000,2010-03-23 15:36:00,0.25,17940,United Kingdom,"2,500.00"
133515,502269,21980,PACK OF 12 RED SPOTTY TISSUES,10000,2010-03-23 15:36:00,0.25,17940,United Kingdom,"2,500.00"


Highest merchandise unit prices:


,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country,sales_value
189991,507637,84016,FLAG OF ST GEORGE CAR FLAG,1,2010-05-10 14:55:00,"1,157.15",<NA>,United Kingdom,"1,157.15"
134868,502451,84016,FLAG OF ST GEORGE CAR FLAG,1,2010-03-24 14:14:00,867.79,<NA>,United Kingdom,867.79
717210,556444,22502,PICNIC BASKET WICKER 60 PIECES,60,2011-06-10 15:28:00,649.50,15098,United Kingdom,"38,970.00"
717221,556446,22502,PICNIC BASKET WICKER 60 PIECES,1,2011-06-10 15:33:00,649.50,15098,United Kingdom,649.50
178910,506571,84016,FLAG OF ST GEORGE CAR FLAG,1,2010-04-30 13:04:00,408.40,<NA>,United Kingdom,408.40
265384,515349,22656,VINTAGE BLUE KITCHEN CABINET,1,2010-07-12 10:43:00,295.00,15513,United Kingdom,295.00
265385,515349,22655,VINTAGE RED KITCHEN CABINET,1,2010-07-12 10:43:00,295.00,15513,United Kingdom,295.00
265625,515403,22655,VINTAGE RED KITCHEN CABINET,1,2010-07-12 11:40:00,295.00,<NA>,United Kingdom,295.00
267389,515602,22656,VINTAGE BLUE KITCHEN CABINET,1,2010-07-13 14:15:00,295.00,14895,United Kingdom,295.00
269044,515811,22656,VINTAGE BLUE KITCHEN CABINET,1,2010-07-15 08:46:00,295.00,15259,United Kingdom,295.00


Highest merchandise line values:


,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country,sales_value
1031554,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2011-12-09 09:15:00,2.08,16446,United Kingdom,"168,469.60"
557400,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,2011-01-18 10:01:00,1.04,12346,United Kingdom,"77,183.60"
717210,556444,22502,PICNIC BASKET WICKER 60 PIECES,60,2011-06-10 15:28:00,649.50,15098,United Kingdom,"38,970.00"
426971,530715,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,9360,2010-11-04 11:36:00,1.69,15838,United Kingdom,"15,818.40"
225363,511465,15044A,PINK PAPER PARASOL,3500,2010-06-08 12:59:00,2.55,18008,United Kingdom,"8,925.00"
842021,567423,23243,SET OF TEA COFFEE SUGAR TINS PANTRY,1412,2011-09-20 11:05:00,5.06,17450,United Kingdom,"7,144.72"
548563,540815,21108,FAIRY CAKE FLANNEL ASSORTED COLOUR,3114,2011-01-11 12:55:00,2.10,15749,United Kingdom,"6,539.40"
655495,550461,21108,FAIRY CAKE FLANNEL ASSORTED COLOUR,3114,2011-04-18 13:20:00,2.10,15749,United Kingdom,"6,539.40"
455824,533027,22086,PAPER CHAIN KIT 50'S CHRISTMAS,835,2010-11-15 16:02:00,6.95,<NA>,United Kingdom,"5,803.25"
375425,525968,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,3120,2010-10-08 10:10:00,1.66,15838,United Kingdom,"5,179.20"


## Investigation of Extreme Merchandise Transactions

The largest quantities, unit prices, and line values are reviewed together with related cancellations and invoice context. Outliers are not automatically removed because legitimate wholesale orders and fully reversed transactions may be present.

In [19]:
suspicious_stock_codes = [
    "23843",
    "23166",
    "22502",
    "84016",
]

outlier_investigation = (
    retail_clean.loc[
        retail_clean["stock_code"].isin(suspicious_stock_codes),
        [
            "invoice_no",
            "stock_code",
            "description",
            "quantity",
            "invoice_date",
            "unit_price",
            "customer_id",
            "country",
            "transaction_type",
            "line_category",
            "line_value",
        ],
    ]
    .sort_values(
        ["stock_code", "customer_id", "invoice_date"],
        na_position="last",
    )
)

display(outlier_investigation)

,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country,transaction_type,line_category,line_value
579758,543370,22502,PICNIC BASKET WICKER SMALL,4,2011-02-07 14:51:00,5.95,12359,Cyprus,Sale,Merchandise,23.80
231133,512063,22502,PICNIC BASKET WICKER SMALL,4,2010-06-13 11:19:00,4.25,12474,Germany,Sale,Merchandise,17.00
543899,540458,22502,PICNIC BASKET WICKER SMALL,4,2011-01-07 12:28:00,5.95,12501,Germany,Sale,Merchandise,23.80
398587,527978,22502,PICNIC BASKET WICKER SMALL,4,2010-10-20 10:45:00,5.95,12523,France,Sale,Merchandise,23.80
491642,535960,22502,PICNIC BASKET WICKER SMALL,4,2010-11-29 12:35:00,5.95,12523,France,Sale,Merchandise,23.80
...,...,...,...,...,...,...,...,...,...,...,...
235093,512435,84016,FLAG OF ST GEORGE CAR FLAG,1,2010-06-15 15:54:00,0.81,<NA>,United Kingdom,Sale,Merchandise,0.81
245814,513439,84016,FLAG OF ST GEORGE CAR FLAG,2400,2010-06-24 14:17:00,0.00,<NA>,United Kingdom,Zero-Price Transaction,Merchandise,0.00
245815,513439,84016,FLAG OF ST GEORGE CAR FLAG,1,2010-06-24 14:17:00,272.27,<NA>,United Kingdom,Sale,Merchandise,272.27
255776,514363,84016,EBAY SALES BY THE BOX.,-7100,2010-07-01 18:07:00,0.00,<NA>,United Kingdom,Stock Adjustment,Merchandise,-0.00


In [20]:
top_value_sales = (
    merchandise_sales
    .nlargest(10, "sales_value")
    .loc[lambda data: data["customer_id"].notna()]
    [
        [
            "invoice_no",
            "stock_code",
            "description",
            "quantity",
            "invoice_date",
            "unit_price",
            "customer_id",
            "sales_value",
        ]
    ]
)

cancellation_lookup = (
    merchandise_cancellations
    .loc[lambda data: data["customer_id"].notna()]
    .assign(
        original_quantity=lambda data: data["quantity"].abs()
    )
    [
        [
            "invoice_no",
            "stock_code",
            "customer_id",
            "original_quantity",
            "unit_price",
            "invoice_date",
            "cancellation_value",
        ]
    ]
)

exact_cancellation_matches = top_value_sales.merge(
    cancellation_lookup,
    left_on=[
        "stock_code",
        "customer_id",
        "quantity",
        "unit_price",
    ],
    right_on=[
        "stock_code",
        "customer_id",
        "original_quantity",
        "unit_price",
    ],
    how="left",
    suffixes=("_sale", "_cancellation"),
)

display(exact_cancellation_matches)

,invoice_no_sale,stock_code,description,quantity,invoice_date_sale,unit_price,customer_id,sales_value,invoice_no_cancellation,original_quantity,invoice_date_cancellation,cancellation_value
0,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2011-12-09 09:15:00,2.08,16446,"168,469.60",C581484,"80,995.00",2011-12-09 09:27:00,"168,469.60"
1,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,2011-01-18 10:01:00,1.04,12346,"77,183.60",C541433,"74,215.00",2011-01-18 10:17:00,"77,183.60"
2,556444,22502,PICNIC BASKET WICKER 60 PIECES,60,2011-06-10 15:28:00,649.50,15098,"38,970.00",<NA>,NaN,NaT,NaN
3,530715,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,9360,2010-11-04 11:36:00,1.69,15838,"15,818.40",<NA>,NaN,NaT,NaN
4,511465,15044A,PINK PAPER PARASOL,3500,2010-06-08 12:59:00,2.55,18008,"8,925.00",<NA>,NaN,NaT,NaN
5,567423,23243,SET OF TEA COFFEE SUGAR TINS PANTRY,1412,2011-09-20 11:05:00,5.06,17450,"7,144.72",<NA>,NaN,NaT,NaN
6,540815,21108,FAIRY CAKE FLANNEL ASSORTED COLOUR,3114,2011-01-11 12:55:00,2.10,15749,"6,539.40",C550456,"3,114.00",2011-04-18 13:08:00,"6,539.40"
7,550461,21108,FAIRY CAKE FLANNEL ASSORTED COLOUR,3114,2011-04-18 13:20:00,2.10,15749,"6,539.40",C550456,"3,114.00",2011-04-18 13:08:00,"6,539.40"
8,525968,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,3120,2010-10-08 10:10:00,1.66,15838,"5,179.20",<NA>,NaN,NaT,NaN


In [21]:
suspicious_invoice_numbers = [
    "507637",
    "502451",
    "556444",
    "556646",
    "506571",
]

suspicious_invoice_context = (
    retail_clean.loc[
        retail_clean["invoice_no"].isin(
            suspicious_invoice_numbers
        ),
        [
            "invoice_no",
            "stock_code",
            "description",
            "quantity",
            "invoice_date",
            "unit_price",
            "customer_id",
            "country",
            "transaction_type",
            "line_value",
        ],
    ]
    .sort_values(["invoice_no", "stock_code"])
)

display(suspicious_invoice_context)

,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country,transaction_type,line_value
134868,502451,84016,FLAG OF ST GEORGE CAR FLAG,1,2010-03-24 14:14:00,867.79,<NA>,United Kingdom,Sale,867.79
134867,502451,POST,POSTAGE,34,2010-03-24 14:14:00,1.00,<NA>,United Kingdom,Sale,34.00
178909,506571,84016,FLAG OF ST GEORGE CAR FLAG,3600,2010-04-30 13:04:00,0.00,<NA>,United Kingdom,Zero-Price Transaction,0.00
178910,506571,84016,FLAG OF ST GEORGE CAR FLAG,1,2010-04-30 13:04:00,408.40,<NA>,United Kingdom,Sale,408.40
189991,507637,84016,FLAG OF ST GEORGE CAR FLAG,1,2010-05-10 14:55:00,"1,157.15",<NA>,United Kingdom,Sale,"1,157.15"
189992,507637,84016,FLAG OF ST GEORGE CAR FLAG,10200,2010-05-10 14:55:00,0.00,<NA>,United Kingdom,Zero-Price Transaction,0.00
717210,556444,22502,PICNIC BASKET WICKER 60 PIECES,60,2011-06-10 15:28:00,649.50,15098,United Kingdom,Sale,"38,970.00"


In [22]:
picnic_basket_investigation = (
    retail_clean.loc[
        retail_clean["stock_code"].eq("22502")
        & retail_clean["customer_id"].eq(15098).fillna(False),
        [
            "invoice_no",
            "stock_code",
            "description",
            "quantity",
            "invoice_date",
            "unit_price",
            "customer_id",
            "transaction_type",
            "line_value",
        ],
    ]
    .sort_values("invoice_date")
)

print("Transactions for customer 15098 and stock code 22502:")
display(picnic_basket_investigation)

price_history_22502 = (
    retail_clean.loc[
        retail_clean["stock_code"].eq("22502")
        & retail_clean["transaction_type"].eq("Sale")
    ]
    .groupby(
        ["description", "unit_price"],
        dropna=False,
        observed=True,
    )
    .agg(
        row_count=("invoice_no", "size"),
        total_quantity=("quantity", "sum"),
        total_value=("line_value", "sum"),
    )
    .reset_index()
    .sort_values("unit_price")
)

print("Complete selling-price history for stock code 22502:")
display(price_history_22502)

Transactions for customer 15098 and stock code 22502:


,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,transaction_type,line_value
717209,556442,22502,PICNIC BASKET WICKER SMALL,60,2011-06-10 15:22:00,4.95,15098,Sale,297.00
717210,556444,22502,PICNIC BASKET WICKER 60 PIECES,60,2011-06-10 15:28:00,649.50,15098,Sale,"38,970.00"
717221,556446,22502,PICNIC BASKET WICKER 60 PIECES,1,2011-06-10 15:33:00,649.50,15098,Sale,649.50
717231,C556448,22502,PICNIC BASKET WICKER SMALL,-60,2011-06-10 15:39:00,4.95,15098,Cancellation,-297.00


Complete selling-price history for stock code 22502:


,description,unit_price,row_count,total_quantity,total_value
1,PICNIC BASKET WICKER SMALL,2.00,1,1,2.00
2,PICNIC BASKET WICKER SMALL,3.75,22,389,"1,458.75"
3,PICNIC BASKET WICKER SMALL,4.25,187,585,"2,486.25"
4,PICNIC BASKET WICKER SMALL,4.95,42,985,"4,875.75"
5,PICNIC BASKET WICKER SMALL,5.95,328,1060,"6,307.00"
6,PICNIC BASKET WICKER SMALL,8.29,96,185,"1,533.65"
7,PICNIC BASKET WICKER SMALL,8.47,154,320,"2,710.40"
8,PICNIC BASKET WICKER SMALL,8.95,1,30,268.50
9,PICNIC BASKET WICKER SMALL,10.79,98,150,"1,618.50"
0,PICNIC BASKET WICKER 60 PIECES,649.50,2,61,"39,619.50"


### Outlier Treatment Decision

Invoice 556444 contains 60 units of a 60-piece picnic-basket package priced at £649.50 per unit, producing a £38,970 line value. Five minutes later, invoice 556446 records one unit of the same package at the same price for the same customer.

This sequence strongly indicates that invoice 556444 contains an erroneous quantity. The row is retained in the standardized dataset for auditability, flagged as a suspected entry error, and excluded from primary revenue and customer analysis. Fully reversed transactions remain in the data and are handled through their signed cancellation values.

In [23]:
retail_clean["data_quality_flag"] = "Standard"
retail_clean["analysis_eligible"] = True

suspected_entry_error = (
    retail_clean["invoice_no"].eq("556444")
    & retail_clean["stock_code"].eq("22502")
    & retail_clean["quantity"].eq(60)
    & retail_clean["unit_price"].eq(649.50)
)

retail_clean.loc[
    suspected_entry_error,
    "data_quality_flag",
] = "Suspected Entry Error"

retail_clean.loc[
    suspected_entry_error,
    "analysis_eligible",
] = False

retail_clean["data_quality_flag"] = (
    retail_clean["data_quality_flag"].astype("category")
)

analysis_merchandise_sales = retail_clean.loc[
    retail_clean["transaction_type"].eq("Sale")
    & retail_clean["line_category"].eq("Merchandise")
    & retail_clean["analysis_eligible"]
].copy()

analysis_merchandise_sales["sales_value"] = (
    analysis_merchandise_sales["line_value"]
)

analysis_merchandise_cancellations = retail_clean.loc[
    retail_clean["transaction_type"].eq("Cancellation")
    & retail_clean["line_category"].eq("Merchandise")
].copy()

excluded_value = retail_clean.loc[
    suspected_entry_error,
    "line_value",
].sum()

adjusted_gross_value = (
    analysis_merchandise_sales["sales_value"].sum()
)

cancellation_value = (
    analysis_merchandise_cancellations["line_value"]
    .abs()
    .sum()
)

adjusted_net_value = (
    adjusted_gross_value
    + analysis_merchandise_cancellations["line_value"].sum()
)

print("Flagged rows:", suspected_entry_error.sum())
print("Excluded suspected value:", f"£{excluded_value:,.2f}")
print("Adjusted gross merchandise value:", f"£{adjusted_gross_value:,.2f}")
print("Merchandise cancellation value:", f"£{cancellation_value:,.2f}")
print("Adjusted net merchandise value:", f"£{adjusted_net_value:,.2f}")

Flagged rows: 1
Excluded suspected value: £38,970.00
Adjusted gross merchandise value: £19,604,891.64
Merchandise cancellation value: £716,462.57
Adjusted net merchandise value: £18,888,429.07


## Final Cleaning Validation

The standardized data and analytical merchandise layers are validated before export. These checks confirm that duplicates have been removed, essential fields are valid, sales contain only positive quantities and prices, cancellations remain separate, and flagged entry errors are excluded from primary analysis.

In [24]:
identified_merchandise_sales = analysis_merchandise_sales.loc[
    analysis_merchandise_sales["customer_id"].notna()
].copy()

standardized_business_columns = [
    "invoice_no",
    "stock_code",
    "description",
    "quantity",
    "invoice_date",
    "unit_price",
    "customer_id",
    "country",
]

remaining_business_duplicates = (
    retail_clean.duplicated(
        subset=standardized_business_columns
    ).sum()
)

validation_summary = pd.DataFrame(
    {
        "Check": [
            "Standardized rows",
            "Remaining business duplicates",
            "Missing invoice numbers",
            "Missing invoice dates",
            "Missing quantities",
            "Missing unit prices",
            "Eligible merchandise sales rows",
            "Identified-customer merchandise rows",
            "Merchandise cancellation rows",
            "Excluded suspected entry errors",
        ],
        "Result": [
            len(retail_clean),
            remaining_business_duplicates,
            retail_clean["invoice_no"].isna().sum(),
            retail_clean["invoice_date"].isna().sum(),
            retail_clean["quantity"].isna().sum(),
            retail_clean["unit_price"].isna().sum(),
            len(analysis_merchandise_sales),
            len(identified_merchandise_sales),
            len(analysis_merchandise_cancellations),
            (~retail_clean["analysis_eligible"]).sum(),
        ],
    }
)

assert remaining_business_duplicates == 0
assert retail_clean["invoice_no"].notna().all()
assert retail_clean["invoice_date"].notna().all()
assert retail_clean["quantity"].notna().all()
assert retail_clean["unit_price"].notna().all()

assert analysis_merchandise_sales["quantity"].gt(0).all()
assert analysis_merchandise_sales["unit_price"].gt(0).all()
assert analysis_merchandise_sales["invoice_no"].str.startswith("C").sum() == 0
assert analysis_merchandise_sales["analysis_eligible"].all()

display(validation_summary)

print("All validation checks passed.")

,Check,Result
0,Standardized rows,1033036
1,Remaining business duplicates,0
2,Missing invoice numbers,0
3,Missing invoice dates,0
4,Missing quantities,0
5,Missing unit prices,0
6,Eligible merchandise sales rows,1003356
7,Identified-customer merchandise rows,776595
8,Merchandise cancellation rows,17915
9,Excluded suspected entry errors,1


All validation checks passed.


## Export Processed Data

The complete standardized transaction dataset is exported as a compressed CSV file for loading into Snowflake. It retains transaction classifications, analytical eligibility, and data-quality flags for traceability.

Processed data files are excluded from GitHub because of their size and can be reproduced by running this notebook.

In [25]:
processed_path = project_root / "data" / "processed"
processed_path.mkdir(parents=True, exist_ok=True)

clean_data_file = (
    processed_path
    / "retail_transactions_clean.csv.gz"
)

validation_file = (
    processed_path
    / "cleaning_validation_summary.csv"
)

export_columns = [
    "invoice_no",
    "stock_code",
    "description",
    "quantity",
    "invoice_date",
    "unit_price",
    "customer_id",
    "country",
    "source_year",
    "transaction_type",
    "line_category",
    "line_value",
    "data_quality_flag",
    "analysis_eligible",
]

retail_clean[export_columns].to_csv(
    clean_data_file,
    index=False,
    compression="gzip",
    date_format="%Y-%m-%d %H:%M:%S",
)

validation_summary.to_csv(
    validation_file,
    index=False,
)

export_manifest = pd.DataFrame(
    {
        "Dataset": [
            "Cleaned transaction data",
            "Cleaning validation summary",
        ],
        "Rows": [
            len(retail_clean),
            len(validation_summary),
        ],
        "File": [
            str(clean_data_file.relative_to(project_root)),
            str(validation_file.relative_to(project_root)),
        ],
        "Size_MB": [
            clean_data_file.stat().st_size / (1024**2),
            validation_file.stat().st_size / (1024**2),
        ],
    }
)

display(export_manifest)
print("Processed data export completed successfully.")

,Dataset,Rows,File,Size_MB
0,Cleaned transaction data,1033036,data\processed\retail_transactions_clean.csv.gz,17.94
1,Cleaning validation summary,10,data\processed\cleaning_validation_summary.csv,0.00


Processed data export completed successfully.


## Final Cleaning Summary

The two Online Retail II worksheets were standardized and combined into one audit-ready transaction dataset.

### Duplicate treatment

The worksheets overlap between 1 and 9 December 2010. Duplicate detection therefore used the underlying transaction fields while excluding the worksheet label from the duplicate key.

- Raw rows: 1,067,371
- Same-source duplicates removed: 12,133
- Cross-source overlap duplicates removed: 22,202
- Total duplicate rows removed: 34,335
- Final standardized rows: 1,033,036
- Remaining business duplicates: 0

### Cleaning and classification

- Standardized column names and data types
- Normalized product descriptions, stock codes, and countries
- Preserved missing customer IDs for overall revenue analysis
- Classified sales, cancellations, stock adjustments, accounting adjustments, and zero-price transactions
- Distinguished merchandise from charges, fees, discounts, vouchers, samples, and manual entries
- Preserved cancellation and adjustment records separately
- Flagged one suspected £38,970 entry error without deleting it from the audit dataset

### Final analytical layers

- Eligible merchandise sales rows: 1,003,356
- Identified-customer merchandise rows: 776,595
- Merchandise cancellation rows: 17,915
- Suspected entry-error rows excluded from analysis: 1

### Revenue validation

- Gross merchandise value before entry-error exclusion: £19,643,861.64
- Suspected entry-error value: £38,970.00
- Adjusted gross merchandise value: £19,604,891.64
- Merchandise cancellation value: £716,462.57
- Adjusted net merchandise value: £18,888,429.07

The corrected standardized dataset was exported as a compressed CSV for Snowflake ingestion. Processed data is excluded from GitHub because it can be reproduced by running this notebook.